# Exercise - Multi step Workflow - STARTER

In this exercise, you’ll build a multi-step workflow using LCEL to solve a more complex task than simply generating a joke. 

**Challenge**

Create an AI Business Advisor that:

1. Accepts an industry as input.
2. Generates a business idea.
3. Analyzes the strengths and weaknesses.
4. Formats the results as a final report.


## 0. Import the necessary libs

In [30]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

## 1. Instantiate Chat Model with your API Key

To be able to connect with OpenAI, you need to instantiate an ChatOpenAI client passing your OpenAI key.

You can pass the `api_key` argument directly.
```python
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key="voc-",
)
```

In [31]:
load_dotenv()
api_key=str(os.getenv("API_KEY"))
base_url=str(os.getenv("BASE_URL"))

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    base_url=base_url,
    temperature=0.7,
)

## 2. Your first Chain

In the end of each chain, you should parse the output and save the logs

In [32]:
logs = []

In [33]:
parser = StrOutputParser()

In [34]:
parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

## 3. Idea Generation

Craft a prompt to generate a business idea for the given industry. 

Make sure {industry} placeholder is inside your template, so it can be filled when the chain is invoked.

In [35]:
# TODO - Your prompt Template
idea_prompt = PromptTemplate(
    template=(
        "Generate business idea about {industry}"
    )
)

In [36]:
# TODO - Create your idea_chain: idea_prompt -> llm -> parse_and_log_output_chain
idea_chain = (idea_prompt | llm | parse_and_log_output_chain)

In [37]:
# TODO - Test your idea_chain invoking it by passing an industy like "agro" to it
idea_result = idea_chain.invoke("agro")

In [38]:
idea_result["output"]

'## Business Idea: **“Smart‑Agri Hub – A Data‑Driven, Community‑Owned Platform for Smallholder Farmers”**\n\n### 1. The Core Concept\nCreate a **digital platform** (mobile app + web portal) that bundles three high‑value services for small‑holder and peri‑urban farmers:\n\n| Service | What It Does | Why It Matters |\n|---------|--------------|----------------|\n| **Precision‑Agri Insights** | AI‑powered crop‑health diagnostics, weather‑forecast integration, soil‑nutrient recommendations, pest‑alert notifications. | Helps farmers increase yields while reducing input waste. |\n| **Marketplace & Micro‑Logistics** | Peer‑to‑peer marketplace for inputs (seeds, bio‑fertilizers, tools) and output sales (produce, milks, eggs) with aggregated, low‑cost transport to local collection points. | Gives farmers better prices, reduces middle‑man squeeze, and solves the “last‑mile” problem. |\n| **Community‑Owned Financing** | A cooperative‑style revolving fund (crowdfunded by local investors, NGOs, imp

In [39]:
logs

[AIMessage(content='## Business Idea: **“Smart‑Agri Hub – A Data‑Driven, Community‑Owned Platform for Smallholder Farmers”**\n\n### 1. The Core Concept\nCreate a **digital platform** (mobile app + web portal) that bundles three high‑value services for small‑holder and peri‑urban farmers:\n\n| Service | What It Does | Why It Matters |\n|---------|--------------|----------------|\n| **Precision‑Agri Insights** | AI‑powered crop‑health diagnostics, weather‑forecast integration, soil‑nutrient recommendations, pest‑alert notifications. | Helps farmers increase yields while reducing input waste. |\n| **Marketplace & Micro‑Logistics** | Peer‑to‑peer marketplace for inputs (seeds, bio‑fertilizers, tools) and output sales (produce, milks, eggs) with aggregated, low‑cost transport to local collection points. | Gives farmers better prices, reduces middle‑man squeeze, and solves the “last‑mile” problem. |\n| **Community‑Owned Financing** | A cooperative‑style revolving fund (crowdfunded by local i

In [40]:
for message in logs:
    print(message.content)

## Business Idea: **“Smart‑Agri Hub – A Data‑Driven, Community‑Owned Platform for Smallholder Farmers”**

### 1. The Core Concept
Create a **digital platform** (mobile app + web portal) that bundles three high‑value services for small‑holder and peri‑urban farmers:

| Service | What It Does | Why It Matters |
|---------|--------------|----------------|
| **Precision‑Agri Insights** | AI‑powered crop‑health diagnostics, weather‑forecast integration, soil‑nutrient recommendations, pest‑alert notifications. | Helps farmers increase yields while reducing input waste. |
| **Marketplace & Micro‑Logistics** | Peer‑to‑peer marketplace for inputs (seeds, bio‑fertilizers, tools) and output sales (produce, milks, eggs) with aggregated, low‑cost transport to local collection points. | Gives farmers better prices, reduces middle‑man squeeze, and solves the “last‑mile” problem. |
| **Community‑Owned Financing** | A cooperative‑style revolving fund (crowdfunded by local investors, NGOs, impact funds)

## 4. Idea Analysis

Craft a prompt to analyze the generated idea's strengths and weaknesses

In [41]:
# TODO - Your prompt Template
analysis_prompt = PromptTemplate(
    template=(
        "Analyse the business idea with list of Strengths and Weaknesses "
        "Idea: {idea}"
        "Identify 3 key strengths and weaknessess of the idea"
    )
)

In [42]:
# TODO - Your chain
analysis_chain = ( analysis_prompt | llm | parse_and_log_output_chain
)

In [43]:
# TODO - Test your analysis_chain invoking it by passing idea_result["output"] to it
analysis_result = analysis_chain.invoke(idea_result["output"])

In [44]:
analysis_result["output"]

'**Three Core Strengths**\n\n| # | Strength | Why it matters |\n|---|----------|----------------|\n| 1️⃣ | **Integrated value‑chain solution** – The platform bundles agronomic advice, a marketplace, and financing in a single flow. | Farmers no longer have to juggle multiple providers; the “one‑stop‑shop” dramatically lowers transaction costs, improves adoption, and creates cross‑selling opportunities that boost lifetime revenue per user. |\n| 2️⃣ | **Community‑owned equity model** – A sizable share of the company is held by farmer cooperatives. | Aligns incentives, builds trust, and makes the business resilient to churn. It also opens doors to impact‑funds and government grants that favor co‑operative or “shared‑value” ventures. |\n| 3️⃣ | **Data‑driven AI diagnostics on low‑bandwidth devices** – Using TensorFlow\u202fLite + USSD fallback lets even the most digitally‑limited farmer access disease‑identification and weather‑adjusted recommendations. | Proven AI models can lift yields 10

In [45]:
logs

[AIMessage(content='## Business Idea: **“Smart‑Agri Hub – A Data‑Driven, Community‑Owned Platform for Smallholder Farmers”**\n\n### 1. The Core Concept\nCreate a **digital platform** (mobile app + web portal) that bundles three high‑value services for small‑holder and peri‑urban farmers:\n\n| Service | What It Does | Why It Matters |\n|---------|--------------|----------------|\n| **Precision‑Agri Insights** | AI‑powered crop‑health diagnostics, weather‑forecast integration, soil‑nutrient recommendations, pest‑alert notifications. | Helps farmers increase yields while reducing input waste. |\n| **Marketplace & Micro‑Logistics** | Peer‑to‑peer marketplace for inputs (seeds, bio‑fertilizers, tools) and output sales (produce, milks, eggs) with aggregated, low‑cost transport to local collection points. | Gives farmers better prices, reduces middle‑man squeeze, and solves the “last‑mile” problem. |\n| **Community‑Owned Financing** | A cooperative‑style revolving fund (crowdfunded by local i

## 5. Report Generation

Craft a prompt to structure the information into a business report.

In [54]:
# TODO - Your prompt Template
report_prompt = PromptTemplate(
    template=(
        "Here is the business analyses:"
        "Strengths and Weaknesses {output}" 
        "Generate a structured report"
    )
)

In [55]:
class AnalysisReport(BaseModel):
    """Strengths and Weaknesses about a business idea"""
    strengths: list = Field(default=[], desc="Idea's strengths")
    weaknesses: list = Field(default=[], desc="Idea's weaknesses")

/tmp/ipykernel_92959/1701761511.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desc'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  strengths: list = Field(default=[], desc="Idea's strengths")
/tmp/ipykernel_92959/1701761511.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desc'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  weaknesses: list = Field(default=[], desc="Idea's weaknesses")


In [56]:
report_chain = (
    report_prompt | llm.with_structured_output(AnalysisReport, method="function_calling")
)

In [57]:
# TODO - Test your report_chain invoking it by passing analysis_result["output"] to it
report_result = report_chain.invoke(analysis_result["output"])

In [58]:
report_result

AnalysisReport(strengths=[{'title': 'Integrated value‑chain solution', 'detail': 'The platform bundles agronomic advice, a marketplace, and financing in a single flow, reducing transaction costs, improving adoption, and enabling cross‑selling opportunities.'}, {'title': 'Community‑owned equity model', 'detail': 'A sizable share of the company is held by farmer cooperatives, aligning incentives, building trust, and opening doors to impact‑funds and government grants.'}, {'title': 'Data‑driven AI diagnostics on low‑bandwidth devices', 'detail': 'TensorFlow\u202fLite + USSD fallback lets even digitally‑limited farmers access disease‑identification and weather‑adjusted recommendations, lifting yields 10‑20\u202f%.'}], weaknesses=[{'title': 'Heavy reliance on seasonal cash‑flow for loan repayment', 'detail': 'Smallholders’ income is concentrated at harvest; crop failures can trigger defaults, threatening the revolving fund and eroding trust.'}, {'title': 'Adoption barrier due to low digital

## 6. End to End Chain

In [59]:
e2e_chain = ( 
    RunnablePassthrough() 
    | idea_chain
    | RunnableParallel(idea=RunnablePassthrough())
    | analysis_chain
    | report_chain
)

In [60]:
e2e_chain.get_graph().print_ascii()

            +------------------+         
            | PassthroughInput |         
            +------------------+         
                      *                  
                      *                  
                      *                  
              +-------------+            
              | Passthrough |            
              +-------------+            
                      *                  
                      *                  
                      *                  
             +----------------+          
             | PromptTemplate |          
             +----------------+          
                      *                  
                      *                  
                      *                  
               +------------+            
               | ChatOpenAI |            
               +------------+            
                      *                  
                      *                  
                      *           

In [61]:
# Change the industry if you want
e2e_result = e2e_chain.invoke("agro")

In [62]:
e2e_result

AnalysisReport(strengths=['Capital‑light, “Farming‑as‑a‑Service” model – By leasing solar‑powered modular pods and bundling them with a SaaS platform, farmers face virtually no upfront CAPEX. This lowers the entry barrier for smallholders and urban growers and creates a recurring‑revenue stream that scales predictably.', 'End‑to‑end data‑driven value chain – The combination of on‑site IoT sensors, AI‑based crop‑health diagnostics, and a marketplace for inputs and produce gives users actionable insights, input‑cost savings, higher yields, and direct market access—all in one platform. The data layer also unlocks ancillary revenue (climate‑credit reporting, aggregated agronomic insights).', 'Off‑grid, climate‑smart design – Built‑in solar panels, battery storage and precision fertigation dramatically reduce water, fertilizer, and energy use while operating in regions with unreliable electricity. This positions the solution well for sustainability‑focused funding, carbon‑credit programs, a

In [63]:
e2e_result.strengths

['Capital‑light, “Farming‑as‑a‑Service” model – By leasing solar‑powered modular pods and bundling them with a SaaS platform, farmers face virtually no upfront CAPEX. This lowers the entry barrier for smallholders and urban growers and creates a recurring‑revenue stream that scales predictably.',
 'End‑to‑end data‑driven value chain – The combination of on‑site IoT sensors, AI‑based crop‑health diagnostics, and a marketplace for inputs and produce gives users actionable insights, input‑cost savings, higher yields, and direct market access—all in one platform. The data layer also unlocks ancillary revenue (climate‑credit reporting, aggregated agronomic insights).',
 'Off‑grid, climate‑smart design – Built‑in solar panels, battery storage and precision fertigation dramatically reduce water, fertilizer, and energy use while operating in regions with unreliable electricity. This positions the solution well for sustainability‑focused funding, carbon‑credit programs, and government subsidies

In [64]:
e2e_result.weaknesses

['Hardware reliability & logistics complexity – Deploying, maintaining, and servicing thousands of modular pods (especially in remote or harsh environments) creates high OPEX and supply‑chain risk. Any downtime directly erodes farmer trust and the recurring‑revenue model.',
 'Adoption friction among low‑tech farmers – Even with leasing and training, smallholder growers may be hesitant to trust AI recommendations, IoT devices, or a subscription‑based service, leading to slower uptake and higher churn than projected.',
 'Competitive pressure from low‑cost DIY kits and specialized SaaS providers – Simpler sensor‑only platforms (e.g., CropX) or inexpensive DIY hydroponic kits can undercut price points. If AgriPulse cannot clearly demonstrate the added value of its bundled hardware‑plus‑service offering, price‑sensitive customers may opt for cheaper alternatives.']

In [65]:
logs

[AIMessage(content='## Business Idea: **“Smart‑Agri Hub – A Data‑Driven, Community‑Owned Platform for Smallholder Farmers”**\n\n### 1. The Core Concept\nCreate a **digital platform** (mobile app + web portal) that bundles three high‑value services for small‑holder and peri‑urban farmers:\n\n| Service | What It Does | Why It Matters |\n|---------|--------------|----------------|\n| **Precision‑Agri Insights** | AI‑powered crop‑health diagnostics, weather‑forecast integration, soil‑nutrient recommendations, pest‑alert notifications. | Helps farmers increase yields while reducing input waste. |\n| **Marketplace & Micro‑Logistics** | Peer‑to‑peer marketplace for inputs (seeds, bio‑fertilizers, tools) and output sales (produce, milks, eggs) with aggregated, low‑cost transport to local collection points. | Gives farmers better prices, reduces middle‑man squeeze, and solves the “last‑mile” problem. |\n| **Community‑Owned Financing** | A cooperative‑style revolving fund (crowdfunded by local i

## 7. Experiment

Now that you understood how it works, experiment with new things.
- Improve memory
- Explore the Runnables
- Add More Complexity